<a href="https://colab.research.google.com/github/Rachel-XMR/Rachel_blog.github.io/blob/Note/Scalable_Data_Engineering_on_the_Cloud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Scalable Data Engineering On The Cloud - Hands-On scenario

Author: João Gabriel Oliveira

Email: joao@jgoliveira.tech

## Setup

### Installing Dependencies

In [ ]:
!pip install pyspark delta-spark

In [ ]:
!apt-get install tree

### Initialization

In [ ]:
import os

In [ ]:
!java --version

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
        .builder
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.3.1")
        .appName("ScalableDataEngnTheCloud")
        .getOrCreate()
)

In [ ]:
DATASET_KEY = "wikimedia-foundation/wikipedia-structured-contents"
ENGLISH_FOLDER_PATH = "enwiki_namespace_0"

In [ ]:
DATA_LAKE_BASE_PATH = "/content/lake"

In [ ]:
use_full_dataset = False
try:
    int(os.getenv("DEOTC_USE_FULL_DATASET"))
    use_full_dataset = True
except (TypeError, ValueError):
    print("DEOTC_USE_FULL_DATASET not set or invalid. A small subset of the dataet will be used.")

In [ ]:
def build_file_name(index):
    return f"{ENGLISH_FOLDER_PATH}_{index}.jsonl"

def build_file_path(index):
    return f"{ENGLISH_FOLDER_PATH}/{build_file_name(index)}"

file_list_to_read = ["*"]
file_path_to_download = None
if not use_full_dataset:
    file_list_to_read = [None]
    file_path_to_download = build_file_path(0)

In [ ]:
file_path_to_download

### Dataset Downloading

In [ ]:
from tqdm import tqdm
import kagglehub

base_file_path = kagglehub.dataset_download(DATASET_KEY, path=file_path_to_download)

In [ ]:
!head $base_file_path


In [ ]:
def get_json_files(target_file_list):
    json_files = []
    for ftr in target_file_list:
        path_parts = [base_file_path, ftr]
        full_file_path = os.path.join(
            *[pp for pp in path_parts if pp is not None]
        )
        json_files.append(f"file:///{full_file_path}")

    return json_files


json_files = get_json_files(file_list_to_read)

df = spark.read.json(json_files)

df.show()

## Pre Analysis

### Partitioning

Apache Spark has built-in support to consume diferent types of data sources. A multitude of third party provided connectors also exist for popular technologies like MongoDB, Elasticsearch, Amazon Redshift, Linux Foundation Delta Tables etc. We will use the later on some of the following sections.

As introduced in the acompaning presentation, a Spark Dataframe is essentially a sequence of instructions on taking data from one place to another, doing some kind of processing in between.

When calling `getNumPartitions` on the underlying RDD we will get **the count of partitions the driver has planned to slice the data into when it is time to go get it**. In other words, while nothing actually happens in terms of data moving around, the driver has already planned how it will slice it when some *action* gets called. So, if we are reading, for instance, a list of 10 files using 5 partitions, the driver might already know that task executor 1 will read files 1 and 2, task executor 2 will read files 3 and 4 and so on.

Now, while no data is actually read, the driver may need to gather information to construct this plan. So, it may need to get statistics on file size, number of file lines, distribution of a given partition key etc.

In [ ]:
df.rdd.getNumPartitions()

Here we count the number of records on each partition:

In [ ]:
from pyspark.sql.functions import spark_partition_id

(
    df.withColumn("partition_id", spark_partition_id())
        .groupBy("partition_id")
        .count()
).show()

### Schema Inference

In addition to partitioning, another critical aspect of data processing involves defining the schema of the input data. There are two possible scenarios for handling the schema:

Explicitly Provided Schema:

In some cases, the schema for the data may be explicitly provided to the read operation. This is often done for structured data where the user knows the exact data types and column names beforehand. By supplying the schema directly, you ensure that the data is interpreted correctly, avoiding any ambiguity or errors that might arise from automatic inference.

Schema Inference:

In scenarios where the schema is not explicitly provided, schema inference can be used instead. This process automatically determines the structure of the data by analyzing a sample of the input data. However, schema inference is not always available; it depends on the type of connector being used. For example, when reading from formats like Parquet or Avro, schema inference is usually built into the connectors. In other cases, such as when reading CSV files, schema inference might involve parsing the data and detecting types based on heuristics, like numeric values being interpreted as integers or floats, and string values being identified as text.

Just as with partitioning decisions, schema inference can introduce performance considerations. When schema inference is enabled, the system may need to scan a portion of the data to determine the structure. This process can add overhead, especially for large datasets, as the driver might need to fetch samples from the source data to perform its analysis. Consequently, it's important to balance the convenience of schema inference with the potential performance impact, particularly in large-scale data processing tasks.

In summary, defining the schema of the input data is a crucial step in data processing. Whether explicitly provided or inferred, the schema plays a central role in ensuring that the data is interpreted correctly and efficiently, allowing subsequent processing tasks to be executed without errors or inefficiencies.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType, TimestampType
from pyspark.sql.functions import col, spark_partition_id

schema = StructType([
    StructField("identifier", IntegerType()),
    StructField("name",StringType()),
    StructField("description",StringType()),
    StructField("date_created", TimestampType()),
    StructField("date_modified", TimestampType()),
    StructField("infoboxes",
        ArrayType(
            StructType(
                [
                    StructField("name", StringType()),
                    StructField("type", StringType()),
                    StructField("has_parts", ArrayType(
                        StructType(
                            [
                                StructField("name", StringType()),
                                StructField("type", StringType()),
                                StructField("has_parts", ArrayType(
                                    StructType(
                                        [
                                            StructField("name", StringType()),
                                            StructField("type", StringType()),
                                            StructField("value", StringType()),
                                        ]
                                    )
                                )),
                            ]
                        )
                    ))
                ]
            )
        )
    ),
])

In [ ]:
df_with_schema = spark.read.schema(schema).json(json_files)

In [ ]:
from pyspark.sql.functions import explode


(
    df_with_schema
        .select(
            "name",
            explode("infoboxes").alias("infoboxes")
            )
        .select(
            "name",
            explode("infoboxes.has_parts").alias("infoboxes_sections")
        ).select(
            "name",
            explode("infoboxes_sections.has_parts").alias("infoboxes_fields")
        ).filter(
            (col("infoboxes_fields.type") == "field") &
            (col("infoboxes_fields.name") == "Occupation")
        ).withColumn("occupation", col("infoboxes_fields.value"))
).show()

## Spark Core APIs

The two main Application Programming Interfaces available in Spark are the RDD API and SparkSQL.

### RDDs

An **RDD (Resilient Distributed Dataset)** is the fundamental data structure in **Apache Spark**. It is a distributed collection of objects, which can be processed in parallel across a cluster of machines.

Key characteristics of an RDD:
- **Resilient**: RDDs are fault-tolerant, meaning they can recover from failures by recomputing lost data using lineage information.
- **Distributed**: The data is split across multiple nodes in a cluster, enabling parallel processing.
- **Immutable**: Once created, RDDs cannot be changed. Instead, transformations (like `map`, `filter`, etc.) create new RDDs.
- **Lazy Evaluation**: RDD operations are lazily evaluated, meaning transformations are not executed until an action (such as `collect` or `count`) is called.

RDDs allow Spark to perform distributed data processing efficiently, supporting both batch and real-time workloads. It uses two broad types of operations:


1. **Transformations**: Operations like `map`, `filter`, and `flatMap` that return a new RDD without altering the original one.
2. **Actions**: Operations like `collect`, `reduce`, and `count` that trigger the execution of the RDD transformations.


In [ ]:
def extract_occupation(row):
    for infobox in row.infoboxes or []:
        for field in infobox.has_parts or []:
            for subfield in field.has_parts or []:
                if subfield.type == 'field' and subfield.name == 'Occupation':
                    occupation_value = subfield.value
                    if occupation_value is not None:
                        return occupation_value

    return None


occupations_rdd = df_with_schema.rdd.map(
    lambda row: (row.name, extract_occupation(row))
).filter(
    lambda row: row[1] is not None
)
occupations_rdd.collect()[:20]


In [ ]:
occupation_points_rdd = df_with_schema.rdd.map(
    lambda row: (extract_occupation(row), 1)
)

occupation_counts = occupation_points_rdd.reduceByKey(lambda x, y: x + y)
occupation_counts.take(20)

In [ ]:
occupations_flat_rdd = occupations_rdd.flatMap(
    lambda x: [(x[0], occ.strip().lower()) for occ in (x[1] or "").split(',') if occ.strip()]
)

occupation_counts = occupations_flat_rdd.map(lambda x: (x[1] ,1)).reduceByKey(lambda x, y: x + y)

occupation_counts.take(20)

### SparkSQL

A **DataFrame** is a distributed collection of data organized into named columns, similar to a table in a relational database or a data frame in R/Pandas. DataFrames in Spark are built on top of RDDs and provide higher-level abstractions for working with structured data.

Key features of DataFrames:
- **Schema**: DataFrames have a schema (i.e., structure) that defines the column names and data types, making it easier to query and manipulate data.
- **Optimized Execution**: DataFrames leverage Spark's **Catalyst optimizer** for query optimization, resulting in faster execution compared to working directly with RDDs.
- **Interoperability**: DataFrames support a wide range of data sources, including JSON, Parquet, Hive, and JDBC, and can be manipulated using both SQL queries and DataFrame operations.
- **Lazy Evaluation**: Like RDDs, DataFrame transformations are lazily evaluated. Computation is only triggered when an action (like `show()`, `collect()`, etc.) is called.

Operations in DataFrames can be one of two types:

- **Transformations**: Operations like `select()`, `filter()`, `groupBy()`, and `join()` return a new DataFrame.
- **Actions**: Operations like `show()`, `collect()`, and `count()` trigger the execution of DataFrame transformations.

The **SparkSQL API** allows you to run SQL queries directly on DataFrames, enabling a familiar interface for users who are comfortable with SQL. It integrates with Spark’s DataFrame API, allowing you to use SQL-style queries to interact with structured data while benefiting from Spark's distributed processing capabilities.

Key features of SparkSQL:
- **SQL Queries on DataFrames**: You can register DataFrames as temporary SQL tables and query them using standard SQL syntax.
- **Integration with Hive**: SparkSQL can query data stored in **Hive** tables and execute SQL queries directly on those.
- **Unified Data Processing**: It allows users to switch seamlessly between DataFrame operations and SQL queries, enabling both developers and analysts to work with the same data using the best tool for the task.


#### `spark.sql.shuffle.partitions`

In Apache Spark, `spark.sql.shuffle.partitions` controls the **number of partitions** created when data is **shuffled** — during operations like:

- `groupBy`
- `join`
- `distinct`
- `repartition`

- **Default value**: `200`
- Applies to **Spark SQL** and **DataFrame** operations.
- **Too many partitions** → small tasks, high overhead.
- **Too few partitions** → underutilized resources, slow execution.

Tune this value based on data size:
- Small dataset? → Use fewer partitions (e.g. 50)
- Large dataset? → Increase partitions to improve parallelism


---


```python
spark.conf.set("spark.sql.shuffle.partitions", 100)


## A Small Data Lake Exercise

In [ ]:
import shutil
from datetime import datetime


def copy_file_to_directory(source_file_path, destination_directory):
    os.makedirs(destination_directory, exist_ok=True)

    filename = os.path.basename(source_file_path)
    destination_file_path = os.path.join(destination_directory, filename)

    try:
        shutil.copy2(source_file_path, destination_file_path)
        print(f"File '{source_file_path}' copied to '{destination_file_path}'")
    except FileNotFoundError:
        print(f"Error: Source file '{source_file_path}' not found.")
    except Exception as e:
        print(f"An error occurred: {e}")

In [ ]:
def build_delta_lake_path(
    base_abs_path,
    item_type,
    format="jsonl",
    version=1,
    use_date_folder_structure=False,
):
    path_parts = [
        base_abs_path,
        f"item_type={item_type}",
        f"format={format}",
        f"version={version}",
    ]

    if use_date_folder_structure:
        current_date = datetime.now()
        year = current_date.year
        month = current_date.month
        day = current_date.day

        path_parts.extend([f"year={year}", f"month={month:02d}", f"day={day:02d}"])

    return os.path.join(
        *path_parts
    )

In [ ]:
target_dl_raw_item_path = build_delta_lake_path(
    DATA_LAKE_BASE_PATH,
    "wikimedia_raw",
    use_date_folder_structure=True
)

copy_file_to_directory(
    base_file_path,
    target_dl_raw_item_path,
)

In [ ]:
from pyspark.sql.functions import current_timestamp


def ingestion_batch(target_files_to_read, schema):
    return spark.read.schema(schema).json(
        get_json_files(target_files_to_read)
    ).withColumn("timestamp", current_timestamp())

In [ ]:
df_with_schema_batch_1 = ingestion_batch(
    file_list_to_read,
    schema
)

df_with_schema_batch_1.show(truncate=False)

In [ ]:
df_with_schema_batch_1.printSchema()

In [ ]:
(
    df_with_schema_batch_1.write
        .mode("append")
        .parquet(
            build_delta_lake_path(
                DATA_LAKE_BASE_PATH, "wikimedia_parsed",
                format="parquet",
            )
        )
)

In [ ]:
df_from_dl_parquet = (
    spark.read
        .parquet(
            build_delta_lake_path(
                DATA_LAKE_BASE_PATH, "wikimedia_parsed",
                format="parquet",
            )
        )
).select("identifier", "name", "timestamp")

In [ ]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window


def get_latest_entries(
    df,
    identifier_col="identifier",
    timestamp_col="timestamp",
):
    window_spec = Window.partitionBy("identifier").orderBy(col("timestamp").desc())
    df_with_row_number = df.withColumn("row_number", row_number().over(window_spec))
    latest_entries_df = df_with_row_number.filter(col("row_number") == 1).drop("row_number")

    return latest_entries_df

get_latest_entries(df_from_dl_parquet).show(truncate=False)

In [ ]:
!tree $DATA_LAKE_BASE_PATH

## Delta Tables

## The Lakehouse/Delta Lake Architecture

### Overview

In [ ]:
def _enable_sparkui(port=4040):
    from google.colab import output
    return output.serve_kernel_port_as_window(port, path='/jobs/index.html')

_enable_sparkui()

In [ ]:
(
    df_with_schema_batch_1.write
        .format("delta")
        .mode("overwrite")
        .option("delta.enableChangeDataFeed", "true")
        .save(
            build_delta_lake_path(
                DATA_LAKE_BASE_PATH, "wikimedia_parsed",
                format="delta",
            )
        )
)

In [ ]:
def latest_changes_for_delta_path(delta_path, starting_version=0):
    return (
        spark.read
            .format("delta")
            .option("readChangeFeed", "true")
            .option("startingVersion", starting_version)
            .load(delta_path)
    ).orderBy(col("_commit_version").desc())

In [ ]:
latest_changes_for_delta_path(
    build_delta_lake_path(
        DATA_LAKE_BASE_PATH, "wikimedia_parsed",
        format="delta",
    )
).show()

In [ ]:
latest_entries_parsed_wiki = get_latest_entries(
    spark.read
        .format("delta")
        .load(
            build_delta_lake_path(
                DATA_LAKE_BASE_PATH, "wikimedia_parsed",
                format="delta",
            )
        )
)

latest_entries_parsed_wiki.show()

In [ ]:
from pyspark.sql.functions import lower


def get_infobox_fields_df(df):
    return (
        latest_entries_parsed_wiki
            .select(
                "identifier",
                "name",
                explode("infoboxes").alias("infoboxes")
                )
            .select(
                "identifier",
                "name",
                explode("infoboxes.has_parts").alias("infoboxes_sections")
            ).select(
                "identifier",
                "name",
                explode("infoboxes_sections.has_parts").alias("infoboxes_fields")
            ).filter(
                (col("infoboxes_fields.type") == "field")
            ).withColumn(
                "field", lower(col("infoboxes_fields.name"))
            ).withColumn(
                "value", lower(col("infoboxes_fields.value"))
            ).select(
                "identifier",
                "name",
                "field",
                "value",
            )
        )

infobox_fields_df = get_infobox_fields_df(latest_entries_parsed_wiki)

infobox_fields_df.show()

In [ ]:
ocupations_df = infobox_fields_df.filter(
    col("field") == "occupation"
)

ocupations_df.show()

In [ ]:
occupations_flat_rdd = ocupations_df.select("identifier", "value").rdd.flatMap(
    lambda x: [(x[0], occ.strip().lower()) for occ in (x[1] or "").split(',') if occ.strip()]
)

occupations_flat_rdd.take(2)

In [ ]:
occupation_entry_df = occupations_flat_rdd.toDF(["identifier", "occupation"])

occupation_entry_df.show(10)

In [ ]:
from pyspark.sql.functions import collect_list

occupation_agg_df = occupation_entry_df.groupBy("identifier").agg(collect_list("occupation").alias("occupations"))

occupation_agg_df.show()


In [ ]:
people_df = latest_entries_parsed_wiki.select(
    "identifier",
    "name",
    "date_created",
    "date_modified",
    "description",
).join(
    occupation_agg_df,
    on="identifier",
    how="inner"
)

people_df.select("identifier", "name", "description").show(100, truncate=False)

In [ ]:
(
    people_df.write
        .format("delta")
        .mode("overwrite")
        .option("delta.enableChangeDataFeed", "true")
        .save(
            build_delta_lake_path(
                DATA_LAKE_BASE_PATH, "wikimedia_people",
                format="delta",
                use_date_folder_structure=False
            )
        )
)

In [ ]:
!tree $DATA_LAKE_BASE_PATH

### Merging Data

In [ ]:
from delta.tables import DeltaTable


deltaTable = DeltaTable.forPath(spark, build_delta_lake_path(
    DATA_LAKE_BASE_PATH, "wikimedia_people",
    format="delta",
    use_date_folder_structure=False
))

In [ ]:
from pyspark.sql.functions import lit

people_updates = deltaTable.toDF().filter(
    col("identifier") == "16520559").withColumn("description", lit("American Writer and activist")
)

In [ ]:
(
    deltaTable.alias("oldData")
        .merge(
            people_updates.alias("newData"),
            "oldData.identifier = newData.identifier"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
)

In [ ]:
latest_changes_for_delta_path(
    build_delta_lake_path(
        DATA_LAKE_BASE_PATH, "wikimedia_people",
        format="delta",
        use_date_folder_structure=False
    )
).show()

### Schema Evolution

In [ ]:
people_df_with_null_col = people_df.withColumn("new_column", lit(None).cast("string"))


(
    people_df_with_null_col.write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .save(
            build_delta_lake_path(
                DATA_LAKE_BASE_PATH, "wikimedia_people",
                format="delta",
                use_date_folder_structure=False
            )
        )
)


### Data Catalog

In [ ]:
(
    people_df_with_null_col.write.format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable("people_on_data_catalog")
)

In [ ]:
spark.sql("""
    SELECT * FROM people_on_data_catalog
""").show()